# Congestion Window Trajectories Under Loss

Companion notebook for the [TCP Flow and Congestion Control](https://codemia.io/courses/networking_for_backend_engineers/net_tcp_flow_and_congestion_control) lesson on Codemia.

**What you will build**: a small simulator for a classic congestion-control algorithm, the sawtooth
it produces, and a comparison against the square-root law that predicts throughput from a loss
rate. By the end you will be able to say what a 0.1% loss rate costs a transfer across an ocean,
without measuring anything.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

MSS = 1460          # bytes per segment

## Concept recap

A sender does not know how much the path can carry, so it probes. It starts small, grows quickly
while nothing is lost, and treats a loss as the signal that it has found the limit. The growth has
two phases. Below a threshold it doubles every round trip, which is slow start and is not slow at
all. Above the threshold it adds roughly one segment per round trip, which is congestion avoidance.

A loss detected by duplicate acknowledgements halves the window and the sender keeps going. A loss
detected by a timer is different twice over: the window restarts at one segment, and the sender
sits idle until the timer fires, which is several round trips of sending nothing at all. The
simulator below charges for both.

## The simulator

In [ ]:
def simulate(rounds=120, loss_rate=0.0, timeout_fraction=0.15, rto_rounds=4,
             initial_cwnd=10 * MSS, ssthresh=64 * MSS, rng=None):
    """One step per round trip. Returns the window at each step, and its phase.

    phase 0 = slow start, 1 = congestion avoidance, 2 = halved after duplicate acknowledgements,
    3 = idle, waiting for the retransmission timer to fire.

    A timer-detected loss appends `rto_rounds` steps with a window of zero: the sender is not
    sending, so those round trips carry nothing and still cost time.
    """
    rng = rng or np.random.default_rng(0)
    cwnd = float(initial_cwnd)
    thresh = float(ssthresh)
    history = []

    while len(history) < rounds:
        segments_in_flight = max(1, int(cwnd // MSS))
        lost = rng.random() < (1 - (1 - loss_rate) ** segments_in_flight)

        if lost:
            thresh = max(2 * MSS, cwnd / 2)
            if rng.random() < timeout_fraction:
                history.extend([(0.0, 3)] * rto_rounds)      # idle until the timer fires
                cwnd = MSS                                   # and start again from one segment
                history.append((cwnd, 0))
            else:
                cwnd = thresh                                # halve and keep sending
                history.append((cwnd, 2))
        else:
            if cwnd < thresh:
                cwnd *= 2
                history.append((cwnd, 0))
            else:
                cwnd += MSS
                history.append((cwnd, 1))

    history = history[:rounds]
    windows = np.array([h[0] for h in history])
    phases = np.array([h[1] for h in history])
    return windows, phases

windows, phases = simulate(rounds=40, loss_rate=0.0)
print('no loss, first 12 round trips, window in segments:')
print([int(w // MSS) for w in windows[:12]])
print('phases:', phases[:12].tolist())

The doubling stops at the threshold and the growth turns linear. Nothing is lost here, so the window
climbs for as long as the transfer lasts, and the only ceiling is the receiver's advertised window.

## The sawtooth

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)

for ax, rate in zip(axes, [0.0001, 0.001, 0.01]):
    w, p = simulate(rounds=300, loss_rate=rate, rng=np.random.default_rng(3))
    ax.plot(w / MSS, linewidth=1.2)
    ax.set_ylabel('cwnd (segments)')
    ax.set_title(f'loss rate {rate:.2%}  (time-average window {np.mean(w)/MSS:,.0f} segments,'
                 f' {np.mean(p == 3):.0%} of round trips idle)')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('round trips')
plt.tight_layout()
plt.show()

Three plots, three regimes. At one loss in ten thousand the sawtooth is wide and the average window
is large. At one in a hundred the window never gets anywhere: every few round trips it is halved or
reset, and the sender spends much of its life recovering rather than sending. The flat stretches at
zero are the timer intervals, and they are pure dead air.

## Throughput from a loss rate, without a simulator

In [ ]:
def mathis_mbps(loss_rate, rtt_ms, mss=MSS):
    """The square-root law: average throughput ~ MSS / (RTT * sqrt(p))."""
    if loss_rate <= 0:
        return float('inf')
    return mss * 8 / (rtt_ms / 1000.0 * np.sqrt(loss_rate)) / 1_000_000

def simulated_mbps(loss_rate, rtt_ms, rounds=4000, seed=11, **kwargs):
    """Bytes delivered divided by elapsed time, where idle round trips count as elapsed."""
    w, _ = simulate(rounds=rounds, loss_rate=loss_rate, rng=np.random.default_rng(seed), **kwargs)
    return np.mean(w) * 8 / (rtt_ms / 1000.0) / 1_000_000

print(f"{'loss':>9}{'RTT':>7}{'formula':>11}{'simulated':>12}")
for rate in [0.00001, 0.0001, 0.001, 0.01]:
    for rtt in [10, 80]:
        print(f'{rate:>9.5f}{rtt:>7}{mathis_mbps(rate, rtt):>10,.1f}{simulated_mbps(rate, rtt):>12,.1f}')

The two columns are within a small factor of each other, which is as much as either deserves. The
formula ignores timeouts, the simulator models them crudely, and the real world has both plus
queueing. What matters is the shape: throughput falls with the square root of the loss rate and
inversely with the round trip. Double the distance and you halve the throughput. Take the loss rate
from 0.01% to 1% and you lose an order of magnitude.

In [ ]:
rates = np.logspace(-5, -1.5, 60)

plt.figure(figsize=(9, 5))
for rtt in [1, 10, 40, 80, 150]:
    plt.plot(rates, [mathis_mbps(r, rtt) for r in rates], label=f'RTT {rtt} ms')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('packet loss rate')
plt.ylabel('throughput (Mbit/s)')
plt.title('A single connection, loss-limited')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

Read the 80ms line at a loss rate of 0.1%: around 4.6 Mbit/s. That is the ceiling for one
connection, on any link, however wide. A transfer that needs more has to reduce the round trip,
reduce the loss, or use more than one connection, and only the third is under the application's own
control.

## Variations: what recovery costs

In [ ]:
print('loss rate 0.2%, 80ms round trip, varying how losses are detected')
print(f"{'timeouts':>10}{'idle':>8}{'mean cwnd':>12}{'Mbit/s':>10}")
for timeout_fraction in [0.0, 0.15, 0.5, 1.0]:
    w, p = simulate(rounds=4000, loss_rate=0.002, timeout_fraction=timeout_fraction,
                    rng=np.random.default_rng(5))
    print(f'{timeout_fraction:>9.0%}{np.mean(p == 3):>8.0%}{np.mean(w)/MSS:>11,.1f}'
          f'{np.mean(w) * 8 / 0.08 / 1_000_000:>10,.1f}')

print()
print('and varying how long the timer waits, at a fixed 50% timeout share')
for rto_rounds in [1, 4, 10, 25]:
    w, _ = simulate(rounds=4000, loss_rate=0.002, timeout_fraction=0.5, rto_rounds=rto_rounds,
                    rng=np.random.default_rng(5))
    print(f'  timer = {rto_rounds:>2} round trips -> {np.mean(w) * 8 / 0.08 / 1_000_000:>6,.1f} Mbit/s')

A loss detected by duplicate acknowledgements costs half the window. A loss detected by a timer
costs that and the waiting, and the second table shows which of the two dominates: the penalty
scales with how long the timer waits, and the minimum retransmission timeout on most stacks is
200ms, which on a short path is dozens of round trips of silence. The simulator here is generous,
because it keeps the loss rate fixed while the timer grows; a real stack also doubles the timer on
each successive failure.

That is the real argument for selective acknowledgement and for anything that keeps the
acknowledgement stream flowing, including sending a little more data than strictly necessary at the
end of a transfer. A single lost packet with nothing behind it to generate duplicates cannot be
detected any other way than by the timer.

## What to try next

- Add a receiver window ceiling and watch the sawtooth flatten into a line. That is what a transfer
  looks like when the window, not the loss, is the constraint, and it is the subject of the
  bandwidth-delay product notebook in this directory.
- Implement the cubic growth function and compare its recovery after a loss against the linear one
  here. The difference is most visible on a long path, which is what it was designed for.
- Model two connections sharing a bottleneck with synchronised losses and see how long they take to
  converge to an equal share. The answer is slower than most people assume.